# 04 — RNA: expressão diferencial com o DESeq2 (R24c, 2026-11-19)

Análise **descritiva** pedida pelo usuário em 2026-11-19 (diário, "[R24c] Plano da análise descritiva"): quais genes e vias diferem
entre as classes no RNA, em especial GS × CIN, a fronteira em que o RNA erra. Não classifica e não entra em nenhuma comparação de
classificação.

- **Ambiente próprio:** kernel "Python (cg-deseq2)" (`requirements_deseq2.txt`). O `pydeseq2` exige pandas < 3, e o `cg-multiomica`
  usa o pandas 3.0.6 (opção (a) do usuário, 2026-11-19).
- **Coorte inteira, sem folds:** o resultado descreve os dados e **não pode escolher genes para nenhum classificador** (seria vazamento:
  os pacientes de teste de cada fold entram no cálculo).
- **Dados:** as contagens brutas do STAR (`unstranded`) dos 336 pacientes, somadas por símbolo nos 17.734 genes do universo do RNA.
- **DESeq2** (`pydeseq2` 0.5.4, padrões do pacote): desenho `~subtipo`, CIN como referência; contrastes GS × CIN (principal), MSI × CIN e
  EBV × CIN. Gene diferencialmente expresso: p ajustado (Benjamini-Hochberg) < 0,05 e |log2 da razão| ≥ 1 (razão sem encolhimento).
- **Vias:** sobre-representação dos genes que sobem e dos que descem nos 52 conjuntos do R25 (teste exato de Fisher unilateral; correção
  de Benjamini-Hochberg nos 104 testes de cada contraste).
- **Ligação com o R24a:** quantos genes do GS × CIN estão nas listas do GS que o R24a selecionou em cada fold.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from scipy.stats import false_discovery_control, fisher_exact

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
INTERIM, PROCESSED, RESULTS = RAIZ / "data" / "interim", RAIZ / "data" / "processed", RAIZ / "results"
TAB, LOGS = RESULTS / "04_models_RNA_deseq2", RESULTS / "logs"  # tabelas deste notebook; logs
for _p in (TAB, LOGS):
    _p.mkdir(parents=True, exist_ok=True)
CLASSES = ["CIN", "MSI", "GS", "EBV"]  # = classe_idx 0, 1, 2, 3
GS, K = CLASSES.index("GS"), 5
PADJ, LFC = 0.05, 1.0  # gene diferencialmente expresso: p ajustado < 0,05 e |log2 da razão| ≥ 1 (plano do R24c)
NUCLEOS = 4
LOG = LOGS / "04_models_RNA_deseq2.log"


def log(msg):
    # como a log() do 04_models_RNA: horário, RAM do processo e pico
    mem = {l.split(":")[0]: int(l.split()[1]) // 1024 for l in open("/proc/self/status") if l.startswith(("VmRSS", "VmHWM"))}
    linha = f"{time.strftime('%H:%M:%S')} [RAM {mem['VmRSS']} MB, pico {mem['VmHWM']} MB] {msg}"
    with open(LOG, "a") as f:
        f.write(linha + "\n")
    print(linha)


log("=== nova execução ===")
coorte = pd.read_csv(PROCESSED / "coorte_cv.csv")
y, fold = coorte.classe_idx.to_numpy(), coorte.fold.to_numpy()
genes = pd.read_csv(INTERIM / "rna_genes.csv", index_col="gene_id")
UNIV = genes.index[genes.no_universo_dna]


def por_simbolo(m):
    # cópia do 04_models_RNA: soma as linhas (Ensembl) de cada símbolo do universo; amostras na ordem do coorte_cv.csv
    return m.loc[UNIV].groupby(genes.gene_name[UNIV]).sum().T.loc[coorte.amostra]


CONT = por_simbolo(pd.read_parquet(INTERIM / "rna_contagens.parquet"))
TPM = por_simbolo(pd.read_parquet(INTERIM / "rna_tpm.parquet"))  # só para refazer as listas do R24a (t de Welch no log2(TPM + 1))
meta = pd.DataFrame({"subtipo": np.array(CLASSES)[y]}, index=coorte.amostra)
assert CONT.shape == TPM.shape == (336, 17_734) and (CONT.columns == TPM.columns).all()
assert (CONT.dtypes == np.int64).all() and (CONT.to_numpy() >= 0).all()  # contagens inteiras, como o DESeq2 exige
print(f"{len(CONT)} pacientes {meta.subtipo.value_counts().loc[CLASSES].to_dict()} · {CONT.shape[1]:,} genes · pandas {pd.__version__}")

19:38:33 [RAM 257 MB, pico 257 MB] === nova execução ===


336 pacientes {'CIN': 202, 'MSI': 59, 'GS': 46, 'EBV': 29} · 17,734 genes · pandas 2.3.3


In [2]:
t0 = time.time()
log(f"=== R24c: DESeq2, {CONT.shape[1]:,} genes, {NUCLEOS} núcleos ===")
dds = DeseqDataSet(counts=CONT, metadata=meta, design="~subtipo", inference=DefaultInference(n_cpus=NUCLEOS), quiet=True)
dds.deseq2()
log(f"DESeq2 ajustado em {(time.time() - t0) / 60:.1f} min")

# autoteste: o fator de escala do pacote é o da mediana das razões, calculado à mão (genes sem zero em nenhum paciente)
k_ = CONT.to_numpy(np.float64)
ref = (k_ > 0).all(axis=0)
s_mao = np.median(k_[:, ref] / np.exp(np.log(k_[:, ref]).mean(axis=0)), axis=1)
assert np.allclose(dds.obs["size_factors"].to_numpy(), s_mao)
print(f"fator de escala: mediana das razões com {ref.sum():,} genes de referência; de {s_mao.min():.2f} a {s_mao.max():.2f}")
del k_

CONTRASTES = {"GS × CIN": "GS", "MSI × CIN": "MSI", "EBV × CIN": "EBV"}
RES = {}
for nome, cl in CONTRASTES.items():
    st = DeseqStats(dds, contrast=["subtipo", cl, "CIN"], alpha=PADJ, inference=DefaultInference(n_cpus=NUCLEOS), quiet=True)
    st.summary()
    RES[nome] = st.results_df
    st.results_df.to_csv(TAB / f"04_models_RNA_r24c_{cl}_x_CIN.csv")
log(f"contrastes calculados; total {(time.time() - t0) / 60:.1f} min")


def de(r, sinal):
    # genes diferencialmente expressos de um contraste: p ajustado < 0,05 e log2 da razão ≥ 1 (sinal +1) ou ≤ −1 (sinal −1)
    return r.index[(r.padj < PADJ) & (sinal * r.log2FoldChange >= LFC)]


resumo = pd.DataFrame({n: {"genes testados (p ajustado não vazio)": int(r.padj.notna().sum()),
                           "p ajustado < 0,05": int((r.padj < PADJ).sum()), "sobem (≥ 1)": len(de(r, 1)), "descem (≤ −1)": len(de(r, -1))}
                       for n, r in RES.items()}).T
print("n por classe:", meta.subtipo.value_counts().loc[CLASSES].to_dict(), "· referência: CIN")
display(resumo)
g = RES["GS × CIN"]
for sinal, txt in ((1, "sobem"), (-1, "descem")):
    print(f"GS × CIN: os 15 genes que mais {txt} no GS (entre os diferencialmente expressos, pelo log2 da razão)")
    display(g.loc[de(g, sinal)].sort_values("log2FoldChange", ascending=sinal < 0).head(15)[["baseMean", "log2FoldChange", "padj"]]
            .round({"baseMean": 0, "log2FoldChange": 2}))

19:38:49 [RAM 899 MB, pico 1077 MB] === R24c: DESeq2, 17,734 genes, 4 núcleos ===


19:41:32 [RAM 1482 MB, pico 1526 MB] DESeq2 ajustado em 2.7 min


fator de escala: mediana das razões com 13,288 genes de referência; de 0.28 a 2.04


19:41:50 [RAM 1484 MB, pico 1629 MB] contrastes calculados; total 3.0 min
n por classe: {'CIN': 202, 'MSI': 59, 'GS': 46, 'EBV': 29} · referência: CIN


,genes testados (p ajustado não vazio),"p ajustado < 0,05",sobem (≥ 1),descem (≤ −1)
GS × CIN,17641,9784,1291,1262
MSI × CIN,17641,9963,144,2853
EBV × CIN,17641,7813,419,2885


GS × CIN: os 15 genes que mais sobem no GS (entre os diferencialmente expressos, pelo log2 da razão)


,baseMean,log2FoldChange,padj
gene_name,,,
GCG,37.0,3.39,6.703209e-07
FCRLA,384.0,3.29,6.572724e-22
RBFOX3,225.0,3.22,7.327175e-17
CLEC17A,162.0,3.16,1.363054e-22
SMYD1,632.0,3.14,8.565269e-09
CD79B,266.0,3.12,8.245705e-30
TCL1A,192.0,3.08,5.248886e-13
CXCR5,35.0,2.91,3.484910e-15
FCER2,189.0,2.90,6.725273e-14


GS × CIN: os 15 genes que mais descem no GS (entre os diferencialmente expressos, pelo log2 da razão)


,baseMean,log2FoldChange,padj
gene_name,,,
CT45A1,33.0,-13.02,2.302554e-05
CT45A10,23.0,-12.73,8.591166e-05
GAGE2A,9.0,-9.80,6.178655e-06
MAGEA4,688.0,-8.55,1.069549e-26
CLDN6,850.0,-6.39,2.440449e-39
MAGEA10,134.0,-6.38,3.247354e-19
MAGEA1,170.0,-6.23,1.122359e-15
ALB,525.0,-6.20,6.388946e-31
PASD1,27.0,-5.96,4.931973e-11


In [3]:
# vias: os símbolos das listas no universo como no R25 (cópia do 04_models_RNA): o próprio; senão, o atual de um símbolo anterior do
# HGNC (anterior de um gene só); senão, um anterior do HGNC que está no universo
vias = pd.read_csv(INTERIM / "vias_genes.csv")
hgnc = pd.read_csv(RAIZ / "data" / "raw" / "hgnc_complete_set.txt", sep="\t", usecols=["symbol", "prev_symbol"], low_memory=False)
ant = hgnc.dropna(subset=["prev_symbol"]).assign(prev=lambda d: d.prev_symbol.str.split("|")).explode("prev")
ant = ant[~ant.prev.duplicated(keep=False)]
ATUAL, ANTERIORES, NO_UNIV = dict(zip(ant.prev, ant.symbol)), ant.groupby("symbol").prev.agg(list), set(CONT.columns)


def no_universo(g):
    if g in NO_UNIV:
        return g
    if ATUAL.get(g) in NO_UNIV:
        return ATUAL[g]
    velhos = [a for a in ANTERIORES.get(g, []) if a in NO_UNIV]
    return velhos[0] if len(velhos) == 1 else None


vias["no_universo"] = vias.gene.map(no_universo)
assert vias.no_universo.notna().sum() == 7_383  # o mesmo do R25
CONJ = {n: set(gr.no_universo.dropna()) for n, gr in vias.groupby("conjunto", sort=False)}


def enriquecimento(r):
    # sobre-representação dos genes que sobem e dos que descem em cada conjunto (Fisher unilateral; universo = genes testados);
    # Benjamini-Hochberg nos 2 × 52 testes do contraste
    testados = set(r.index[r.padj.notna()])
    linhas = {}
    for sinal, txt in ((1, "sobem"), (-1, "descem")):
        d = set(de(r, sinal)) & testados
        for n, s in CONJ.items():
            s = s & testados
            a = len(d & s)
            tab = [[a, len(d) - a], [len(s) - a, len(testados) - len(d) - len(s) + a]]
            res = fisher_exact(tab, alternative="greater")
            linhas[(txt, n)] = {"genes do conjunto": len(s), "diferenciais no conjunto": a, "fração": a / len(s),
                                "razão de chances": res.statistic, "p": res.pvalue}
    t = pd.DataFrame(linhas).T.rename_axis(["direção", "conjunto"])
    t["p ajustado"] = false_discovery_control(t.p.to_numpy(float))
    return t


# autoteste do teste de Fisher à mão: 2 de 2 genes diferenciais no conjunto de 2, com 4 testados → p = C(2,2)·C(2,0) / C(4,2) = 1/6
assert np.isclose(fisher_exact([[2, 0], [0, 2]], alternative="greater").pvalue, 1 / 6)

ENR = {n: enriquecimento(r) for n, r in RES.items()}
pd.concat(ENR, names=["contraste"]).to_csv(TAB / "04_models_RNA_r24c_vias.csv")
print("Conjuntos com p ajustado < 0,05, por contraste e direção:")
display(pd.DataFrame({n: (t["p ajustado"] < PADJ).groupby(level="direção").sum() for n, t in ENR.items()}).T)
COLS = ["genes do conjunto", "diferenciais no conjunto", "fração", "razão de chances", "p ajustado"]
e = ENR["GS × CIN"]
print("GS × CIN: conjuntos sobre-representados (p ajustado < 0,05), em ordem de p ajustado")
display(e[e["p ajustado"] < PADJ].sort_values("p ajustado")[COLS].astype({"genes do conjunto": int, "diferenciais no conjunto": int})
        .round({"fração": 3, "razão de chances": 1}))
log("vias terminadas")

Conjuntos com p ajustado < 0,05, por contraste e direção:


direção,descem,sobem
GS × CIN,6,8
MSI × CIN,4,8
EBV × CIN,2,10


GS × CIN: conjuntos sobre-representados (p ajustado < 0,05), em ordem de p ajustado


genes do conjunto  \
direção conjunto                                                        
sobem   HALLMARK_MYOGENESIS                                       195   
        Stromal141_UP                                             133   
        Immune141_UP                                              132   
        HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION                195   
descem  HALLMARK_G2M_CHECKPOINT                                   196   
        HALLMARK_SPERMATOGENESIS                                  132   
        HALLMARK_E2F_TARGETS                                      197   
sobem   HALLMARK_APICAL_JUNCTION                                  193   
        HALLMARK_UV_RESPONSE_DN                                   144   
descem  HALLMARK_KRAS_SIGNALING_DN                                187   
        HALLMARK_COAGULATION                                      132   
sobem   HALLMARK_ALLOGRAFT_REJECTION                              191   
        HALLMARK_KRAS_SIGNALING_DN                                187   
descem  HALLMARK_PANCREAS_BETA_CELLS                               40   

                                                    diferenciais no conjunto  \
direção conjunto                                                               
sobem   HALLMARK_MYOGENESIS                                               54   
        Stromal141_UP                                                     42   
        Immune141_UP                                                      41   
        HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION                        40   
descem  HALLMARK_G2M_CHECKPOINT                                           38   
        HALLMARK_SPERMATOGENESIS                                          26   
        HALLMARK_E2F_TARGETS                                              31   
sobem   HALLMARK_APICAL_JUNCTION                                          30   
        HALLMARK_UV_RESPONSE_DN                                           24   
descem  HALLMARK_KRAS_SIGNALING_DN                                        27   
        HALLMARK_COAGULATION                                              21   
sobem   HALLMARK_ALLOGRAFT_REJECTION                                      25   
        HALLMARK_KRAS_SIGNALING_DN                                        24   
descem  HALLMARK_PANCREAS_BETA_CELLS                                       8   

                                                    fração  razão de chances  \
direção conjunto                                                               
sobem   HALLMARK_MYOGENESIS                          0.277               5.0   
        Stromal141_UP                                0.316               6.0   
        Immune141_UP                                 0.311               5.9   
        HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION   0.205               3.3   
descem  HALLMARK_G2M_CHECKPOINT                      0.194               3.2   
        HALLMARK_SPERMATOGENESIS                     0.197               3.2   
        HALLMARK_E2F_TARGETS                         0.157               2.5   
sobem   HALLMARK_APICAL_JUNCTION                     0.155               2.4   
        HALLMARK_UV_RESPONSE_DN                      0.167               2.6   
descem  HALLMARK_KRAS_SIGNALING_DN                   0.144               2.2   
        HALLMARK_COAGULATION                         0.159               2.5   
sobem   HALLMARK_ALLOGRAFT_REJECTION                 0.131               1.9   
        HALLMARK_KRAS_SIGNALING_DN                   0.128               1.9   
descem  HALLMARK_PANCREAS_BETA_CELLS                 0.200               3.3   

                                                      p ajustado  
direção conjunto                                                  
sobem   HALLMARK_MYOGENESIS                         4.199293e-16  
        Stromal141_UP                               6.346624e-15  
        Immune141_UP                                1

19:41:55 [RAM 1559 MB, pico 1629 MB] vias terminadas


In [4]:
def t_welch(x, y_, c):  # cópia do 04_models_RNA (R24a)
    a, b = x[y_ == c], x[y_ != c]
    with np.errstate(divide="ignore", invalid="ignore"):
        return (a.mean(axis=0) - b.mean(axis=0)) / np.sqrt(a.var(axis=0, ddof=1) / len(a) + b.var(axis=0, ddof=1) / len(b))


def selecionar(x_tr, y_tr, n=500):  # cópia do 04_models_RNA (R24a)
    listas = []
    for c in range(len(CLASSES)):
        s = np.abs(t_welch(x_tr, y_tr, c))
        listas.append(np.argsort(-np.where(np.isnan(s), -1, s), kind="stable")[:n])
    return np.unique(np.concatenate(listas)), listas


# as listas do GS do R24a, refeitas no treino de cada fold; o t não muda com o z por gene do R24a, então vem do log2(TPM + 1) direto
X = np.log2(TPM.to_numpy(np.float64) + 1)
SEL = {k: selecionar(X[fold != k], y[fold != k]) for k in range(K)}
assert [len(u) for u, _ in SEL.values()] == [1731, 1725, 1737, 1717, 1730]  # as uniões do R24a (04_models_RNA, 2026-11-19)
LISTAS_GS = {k: set(CONT.columns[ls[GS]]) for k, (_, ls) in SEL.items()}
DE_GS = {"sobem": set(de(g, 1)), "descem": set(de(g, -1))}
print(f"GS × CIN: {len(DE_GS['sobem'])} genes sobem e {len(DE_GS['descem'])} descem no GS")
display(pd.DataFrame({k: {"genes na lista do GS (R24a)": len(l),
                          **{f"GS × CIN, {t}, na lista": len(l & d) for t, d in DE_GS.items()},
                          "lista do GS que é diferencial no GS × CIN": len(l & (DE_GS["sobem"] | DE_GS["descem"])) / len(l)}
                      for k, l in LISTAS_GS.items()}).T.rename_axis("fold").round(3))
uniao_gs = set().union(*LISTAS_GS.values())
todos = DE_GS["sobem"] | DE_GS["descem"]
print(f"Genes diferenciais do GS × CIN em alguma das 5 listas do GS: {len(todos & uniao_gs)} de {len(todos)} "
      f"({len(todos & uniao_gs) / max(len(todos), 1):.0%}); as 5 listas juntas têm {len(uniao_gs)} genes")
log("fim do R24c")

GS × CIN: 1291 genes sobem e 1262 descem no GS


,genes na lista do GS (R24a),"GS × CIN, sobem, na lista","GS × CIN, descem, na lista",lista do GS que é diferencial no GS × CIN
fold,,,,
0,500.0,58.0,133.0,0.382
1,500.0,96.0,116.0,0.424
2,500.0,80.0,139.0,0.438
3,500.0,45.0,128.0,0.346
4,500.0,61.0,146.0,0.414


Genes diferenciais do GS × CIN em alguma das 5 listas do GS: 311 de 2553 (12%); as 5 listas juntas têm 858 genes
19:41:58 [RAM 1653 MB, pico 1685 MB] fim do R24c


### Leitura do R24c (execução de 2026-11-19, 18:51 a 18:55)

Descritivo, sem desfecho de classificação; coorte inteira (não serve para escolher genes de classificador).
- **Quantos genes:** dos 17.641 testados, no GS × CIN 1.291 sobem e 1.262 descem (p ajustado < 0,05 e |log2 da razão| ≥ 1); 9.784 têm p ajustado
  < 0,05 com qualquer razão (com 336 pacientes, diferenças pequenas já dão p baixo; o corte na razão é o que separa). MSI × CIN: 144 sobem e 2.853
  descem; EBV × CIN: 419 sobem e 2.885 descem.
- **Os que mais sobem no GS:** genes de linfócito B (FCRLA, CD79B, TCL1A, CXCR5, FCER2, CLEC17A) e de músculo liso (ACTG2, SMYD1, LDB3).
  **Os que mais descem:** antígenos de câncer-testículo (CT45A1, CT45A10, GAGE2A, MAGEA4, MAGEA10, MAGEA1, SAGE1, CTAG2, PASD1), CLDN6, ALB e IGF2BP1.
  Os log2 da razão extremos (−13 no CT45A1) vêm de genes com contagem média baixa e razão sem encolhimento.
- **Vias (Fisher, p ajustado < 0,05):** sobem no GS miogênese, estroma e imune do ESTIMATE, EMT, junção apical, alorrejeição; descem G2M, E2F,
  espermatogênese, coagulação. É a direção da hipótese do R25 e do descritivo do R25 (estroma mais alto, ciclo celular mais baixo no GS).
- **Ressalva:** o GS tem menos pureza (mais estroma) que o CIN; genes de estroma e de células imunes que sobem no GS refletem em parte a composição
  da amostra, não só o tumor. O DESeq2 aqui não ajusta pela pureza (plano do R24c); ARAN 2015 (`referencias_literatura.md`, §11.3) descreve esse
  confundimento na expressão diferencial.
- **Ligação com o R24a:** 35% a 44% dos 500 genes da lista do GS de cada fold são diferenciais no GS × CIN; dos 2.553 diferenciais do GS × CIN, só 311
  (12%) estão em alguma das 5 listas (858 genes ao todo), porque a lista do R24a compara o GS com as outras 3 classes juntas, não só com o CIN.
- Custo: DESeq2 de 336 × 17.734 em 2,2 min (4 núcleos); contrastes e vias 0,4 min; notebook inteiro 3,3 min; RAM do processo até 1.688 MB.